# 3e. Survival Targets and Stratified Splits
This notebook processes the `interim_PT` cohorts, computes per-cohort median survival for OS/PFS to create binary targets, and generates 10 robust stratified train/test splits.

In [ ]:
# Install local package to get loguru, anndata, iterative-stratification, etc.
!pip install -q -e .

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import sys
from pathlib import Path
from loguru import logger

# Add project root to path if running locally
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# When running in Colab, the root is /content/batchcor-rna-embeds
if "/content/batchcor-rna-embeds" not in sys.path:
    sys.path.insert(0, "/content/batchcor-rna-embeds")

from batchcor_rna_emb.config import *
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr
from batchcor_rna_emb.survival_targets import add_survival_targets
from batchcor_rna_emb.split_utils import generate_stratified_splits

In [ ]:
N_SPLITS = 10
TEST_SIZE = 0.2

for zarr_path in sorted(DATA_INTERIM_PT_DIR.glob("*.adata.zarr")):
    cohort_name = zarr_path.name.replace(".adata.zarr", "")
    logger.info("\n" + "=" * 70)
    logger.info("Processing: {}", cohort_name)
    logger.info("=" * 70)

    try:
        adata = load_cohort(zarr_path)
    except Exception as e:
        logger.warning("Failed to load {}, skipping. Error: {}", cohort_name, e)
        continue

    logger.info("Loaded: {} samples x {} genes", adata.n_obs, adata.n_vars)

    # 1. Add Survival Targets
    add_survival_targets(adata)

    # 2. Generate Stratified Splits
    stratify_cols = [
        DIAGNOSIS_COL, 
        BATCH_COL, 
        "Target_OS_bin", 
        "Target_PFS_bin", 
        "Response"
    ]
    generate_stratified_splits(
        adata,
        stratify_cols=stratify_cols,
        n_splits=N_SPLITS,
        test_size=TEST_SIZE
    )

    # 3. Save back to interim_PT
    logger.info("Saving updated cohort back to {}", zarr_path)
    save_adata_zarr(adata, zarr_path)
    logger.info("Saved successfully.")